## **Projeto:** Merca Data Platform — Propensao a Cancelamento
### Treino Baseline - LightGBM
### Objetivo deste notebook
Treinar o primeiro modelo funcional (baseline), sem otimizacao de hiperparametros - so os parametros ja desenhados e justificados anteriormente. Objetivo e validar que o fluxo completo (dado -> feature
-> modelo -> metrica) funciona de ponta a ponta.
### Por que split temporal, nao aleatorio
As features de historico do cliente (qtd_pedidos_anteriores, taxa_cancelamento_historica) dependem da ordem no tempo. Um split aleatorio misturaria pedidos "futuros" no treino e daria uma metrica de validacao artificialmente otimista - o modelo nunca vai prever o passado em producao, entao a validacao precisa simular isso.

In [0]:
 %pip install lightgbm

In [0]:
%run ../utils/feat_squad2_99_helpers

In [0]:
CAMADA_FEATURES = "ml_features"
TABELA_FEATURES = "propensao_cancelamento_pedidos_v1"
 
df = ler_delta(CAMADA_FEATURES, TABELA_FEATURES)
log.info(f"Total de linhas lidas: {df.count():,}")
df.printSchema()

In [0]:
from pyspark.sql.functions import col, unix_timestamp
 
# approxQuantile nao aceita TimestampType diretamente - convertemos para
# timestamp Unix (numero) so para calcular o corte, e comparamos usando
# a mesma conversao no filtro.
df_com_unix = df.withColumn("dt_pedido_unix", unix_timestamp(col("dt_pedido")))
 
corte_unix = df_com_unix.approxQuantile("dt_pedido_unix", [0.8], 0.01)[0]
 
log.info(f"Corte (timestamp unix, 80o percentil): {corte_unix}")
 
df_treino    = df.filter(unix_timestamp(col("dt_pedido")) <= corte_unix)
df_validacao = df.filter(unix_timestamp(col("dt_pedido")) > corte_unix)
 
qtd_treino    = df_treino.count()
qtd_validacao = df_validacao.count()
 
log.info(f"Treino    : {qtd_treino:,} pedidos ({round(100*qtd_treino/(qtd_treino+qtd_validacao),1)}%)")
log.info(f"Validacao : {qtd_validacao:,} pedidos ({round(100*qtd_validacao/(qtd_treino+qtd_validacao),1)}%)")
 
pos_treino    = df_treino.filter(col("target_cancelamento") == 1).count()
pos_validacao = df_validacao.filter(col("target_cancelamento") == 1).count()
 
log.info(f"Positivos no treino    : {pos_treino:,} ({round(100*pos_treino/qtd_treino,2)}%)")
log.info(f"Positivos na validacao : {pos_validacao:,} ({round(100*pos_validacao/qtd_validacao,2)}%)")

In [0]:
FEATURE_COLS = [
    "valor_total",
    "valor_frete",
    "razao_frete_valor",
    "metodo_pagamento_idx",
    "dia_semana_pedido",
    "mes_pedido",
    "hora_pedido",
    "qtd_pedidos_anteriores",
    "qtd_cancelamentos_anteriores",
    "taxa_cancelamento_historica",
]
TARGET_COL = "target_cancelamento"
 
# Cast explicito para double antes do toPandas - colunas decimal do Spark
# viram objetos Decimal em pandas se nao forem convertidas, e o LightGBM
# nao aceita esse tipo diretamente.
for c in ["valor_total", "valor_frete", "razao_frete_valor"]:
    df_treino    = df_treino.withColumn(c, col(c).cast("double"))
    df_validacao = df_validacao.withColumn(c, col(c).cast("double"))
 
treino_pd    = df_treino.select(*FEATURE_COLS, TARGET_COL).toPandas()
validacao_pd = df_validacao.select(*FEATURE_COLS, TARGET_COL).toPandas()
 
X_treino, y_treino       = treino_pd[FEATURE_COLS], treino_pd[TARGET_COL]
X_validacao, y_validacao = validacao_pd[FEATURE_COLS], validacao_pd[TARGET_COL]
 
log.info(f"X_treino: {X_treino.shape}   X_validacao: {X_validacao.shape}")

In [0]:
scale_pos_weight = round((y_treino == 0).sum() / (y_treino == 1).sum(), 4)
log.info(f"scale_pos_weight (calculado no treino): {scale_pos_weight}")

In [0]:
import lightgbm as lgb
 
params = {
    "objective": "binary",
    "metric": "average_precision",
    "boosting_type": "gbdt",
    "num_leaves": 31,
    "max_depth": 5,
    "min_data_in_leaf": 20,
    "learning_rate": 0.05,
    "lambda_l2": 1.0,
    "min_split_gain": 0.01,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 1,
    "scale_pos_weight": scale_pos_weight,
    "random_state": 42,
    "verbosity": -1,
}
 
modelo = lgb.LGBMClassifier(**params, n_estimators=1000)
 
modelo.fit(
    X_treino, y_treino,
    eval_set=[(X_validacao, y_validacao)],
    eval_metric="average_precision",
    callbacks=[
        lgb.early_stopping(stopping_rounds=50),
        lgb.log_evaluation(period=50),
    ]
)
 
log.info(f"Treino concluido. Melhor iteracao: {modelo.best_iteration_}")

In [0]:
from sklearn.metrics import (
    roc_auc_score, average_precision_score, confusion_matrix,
    precision_recall_curve, f1_score, classification_report
)
import numpy as np
 
y_pred_proba = modelo.predict_proba(X_validacao)[:, 1]
 
auc_roc = roc_auc_score(y_validacao, y_pred_proba)
auc_pr  = average_precision_score(y_validacao, y_pred_proba)
 
log.info(f"ROC-AUC       : {round(auc_roc, 4)}")
log.info(f"PR-AUC (AP)   : {round(auc_pr, 4)}")
 
# Matriz de confusao com limiar padrao 0.5 (referencia, nao a decisao final)
y_pred_05 = (y_pred_proba >= 0.5).astype(int)
log.info("Matriz de confusao (limiar = 0.5):")
print(confusion_matrix(y_validacao, y_pred_05))
print(classification_report(y_validacao, y_pred_05, digits=3))

In [0]:
precisions, recalls, thresholds = precision_recall_curve(y_validacao, y_pred_proba)
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-10)
 
melhor_idx       = np.argmax(f1_scores[:-1])  # ultimo ponto nao tem threshold correspondente
melhor_threshold = thresholds[melhor_idx]
melhor_f1        = f1_scores[melhor_idx]
 
log.info(f"Melhor limiar (maximiza F1) : {round(melhor_threshold, 4)}")
log.info(f"F1 nesse limiar             : {round(melhor_f1, 4)}")
log.info(f"Precisao nesse limiar       : {round(precisions[melhor_idx], 4)}")
log.info(f"Recall nesse limiar         : {round(recalls[melhor_idx], 4)}")
 
y_pred_otimo = (y_pred_proba >= melhor_threshold).astype(int)
log.info("Matriz de confusao (limiar otimo):")
print(confusion_matrix(y_validacao, y_pred_otimo))

In [0]:
import pandas as pd
 
df_importancia = pd.DataFrame({
    "feature": FEATURE_COLS,
    "importancia": modelo.feature_importances_
}).sort_values("importancia", ascending=False)
 
log.info("Importancia das features (baseline, sem tuning):")
print(df_importancia.to_string(index=False))
 

In [0]:
import pickle
 
modelo_bytes = pickle.dumps(modelo)
 
container_client_squad2 = get_squad2_client()
caminho_modelo = "ml_features/modelos/propensao_cancelamento_baseline_v1.pkl"
 
file_client = container_client_squad2.get_file_client(caminho_modelo)
file_client.upload_data(modelo_bytes, overwrite=True)
 
log.info(f"Modelo salvo em: abfss://squad2@{ADLS_STORAGE_ACCOUNT}.dfs.core.windows.net/{caminho_modelo}")
log.info("BASELINE CONCLUIDO.")